In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import uuid

CATALOG = "credit_risk_analysis_db"

BRONZE_SCHEMA = "bronze_sch"

SILVER_SCHEMA = "silver_sch"

In [0]:
tables = {

"applicant_profiles":"applicant_id",

"credit_application":"application_id",

"credit_history":"history_id",

"loan_details":"loan_id",

"economic_indicators":"indicator_id"

}

In [0]:
df = spark.table(
    "credit_risk_analysis_db.bronze_sch.bronze_applicant_profiles"
)

display(df)

In [0]:
silver_df = (

df

.dropDuplicates()

.filter(col("applicant_id").isNotNull())

.withColumn(
"gender",
upper(trim(col("gender")))
)

.withColumn(
"age",
col("age").cast("int")
)

.withColumn(
"income",
col("income").cast("double")
)

.withColumn(
"region",
upper(trim(col("region")))
)

)

In [0]:
silver_df = silver_df.withColumn(

"applicant_sk",

md5(col("applicant_id"))

)

In [0]:
silver_df = (

silver_df

.withColumn("effective_from", current_timestamp())

.withColumn("effective_to", lit(None).cast("timestamp"))

.withColumn("is_current", lit(True))

)

In [0]:
df.select("age").distinct().show(100, False)

In [0]:
df.select("income").distinct().show(100, False)

In [0]:
df.select("total_units").distinct().show(100, False)

In [0]:
df.withColumn(
    "total_units",
    regexp_extract(col("total_units"), r"(\d+)", 1).cast("int")
)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
CATALOG = "credit_risk_analysis_db"

BRONZE_SCHEMA = "bronze_sch"

SILVER_SCHEMA = "silver_sch"

In [0]:
df = spark.table(
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_applicant_profiles"
)

display(df)

In [0]:
silver_df = (

df

.dropDuplicates()

.filter(col("applicant_id").isNotNull())

.withColumn("gender",
            upper(trim(col("gender"))))

.withColumn("age_group",
            trim(col("age")))

.drop("age")

.withColumn("income",
            col("income").cast("double"))

.withColumn("business_or_commercial",
            upper(trim(col("business_or_commercial"))))

.withColumn("occupancy_type",
            upper(trim(col("occupancy_type"))))

.withColumn("construction_type",
            upper(trim(col("construction_type"))))

.withColumn(
    "total_units",
    regexp_extract(col("total_units"), r"(\d+)", 1).cast("int")
)

.withColumn("region",
            upper(trim(col("region")))

))

In [0]:
silver_df = silver_df.withColumn(

"applicant_sk",

md5(col("applicant_id"))

)

In [0]:
silver_df = (

silver_df

.withColumn("effective_from",
            current_timestamp())

.withColumn("effective_to",
            lit(None).cast("timestamp"))

.withColumn("is_current",
            lit(True))

)

In [0]:
(

silver_df.write

.format("delta")

.mode("overwrite")

.option("mergeSchema","true")

.saveAsTable(

f"{CATALOG}.{SILVER_SCHEMA}.silver_applicant_profiles"

)

)

In [0]:
display(

spark.table(

f"{CATALOG}.{SILVER_SCHEMA}.silver_applicant_profiles"

)

)

In [0]:
%sql
OPTIMIZE credit_risk_analysis_db.silver_sch.silver_applicant_profiles;

In [0]:
%sql
OPTIMIZE credit_risk_analysis_db.silver_sch.silver_applicant_profiles

ZORDER BY(applicant_id);

In [0]:
%sql
VACUUM credit_risk_analysis_db.silver_sch.silver_applicant_profiles RETAIN 168 HOURS;

In [0]:
%sql
SELECT *

FROM credit_risk_analysis_db.silver_sch.silver_applicant_profiles

VERSION AS OF 1;

In [0]:
## next file

In [0]:
from pyspark.sql.functions import *

credit_app = spark.table(
    "credit_risk_analysis_db.bronze_sch.bronze_credit_application"
)

display(credit_app)

In [0]:
credit_app.printSchema()

In [0]:
for col_name in credit_app.columns:
    print(f"\n===== {col_name} =====")
    credit_app.select(col_name).distinct().show(20, False)

In [0]:
display(credit_app.limit(10))

In [0]:
credit_app.printSchema()

In [0]:
from pyspark.sql.functions import *

credit_app = spark.table(
    "credit_risk_analysis_db.bronze_sch.bronze_credit_application"
)

In [0]:
cols = [
    "loan_amount",
    "rate_of_interest",
    "Interest_rate_spread",
    "Upfront_charges",
    "term",
    "year"
]

for c in cols:
    print(f"\n===== {c} =====")
    credit_app.select(c).distinct().show(20, False)

In [0]:
silver_credit_application = (

    credit_app

    .dropDuplicates()

    .filter(col("applicant_id").isNotNull())

    .withColumn("loan_limit", upper(trim(col("loan_limit"))))

    .withColumn("approv_in_adv", upper(trim(col("approv_in_adv"))))

    .withColumn("loan_type", upper(trim(col("loan_type"))))

    .withColumn("loan_purpose", upper(trim(col("loan_purpose"))))

    .withColumn("submission_of_application",
                upper(trim(col("submission_of_application"))))

    .withColumn("region",
                upper(trim(col("region"))))

    .withColumn("application_status",
                upper(trim(col("application_status"))))

    .withColumn("year",
                col("year").cast("int"))

    .withColumn("loan_amount",
                col("loan_amount").cast("double"))

    .withColumn("rate_of_interest",
                col("rate_of_interest").cast("double"))

    .withColumn("Interest_rate_spread",
                col("Interest_rate_spread").cast("double"))

    .withColumn("Upfront_charges",
                col("Upfront_charges").cast("double"))

    .withColumn("term",
                col("term").cast("int"))

    .withColumn(
        "application_sk",
        md5(col("applicant_id"))
    )

    .withColumn(
        "effective_from",
        current_timestamp()
    )

    .withColumn(
        "effective_to",
        lit(None).cast("timestamp")
    )

    .withColumn(
        "is_current",
        lit(True)
    )

)

In [0]:
silver_credit_application.printSchema()

display(silver_credit_application)

In [0]:
(
    silver_credit_application.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(
        "credit_risk_analysis_db.silver_sch.silver_credit_application"
    )
)

In [0]:
display(
    spark.table(
        "credit_risk_analysis_db.silver_sch.silver_credit_application"
    )
)

In [0]:
%sql
OPTIMIZE credit_risk_analysis_db.silver_sch.silver_credit_application;

In [0]:
%sql
OPTIMIZE credit_risk_analysis_db.silver_sch.silver_credit_application
ZORDER BY(applicant_id);

In [0]:
%sql
VACUUM credit_risk_analysis_db.silver_sch.silver_credit_application RETAIN 168 HOURS;

In [0]:
#####3

In [0]:
from pyspark.sql.functions import *

credit_history = spark.table(
    "credit_risk_analysis_db.bronze_sch.bronze_credit_history"
)

display(credit_history)

In [0]:
credit_history.printSchema()

In [0]:
for c in credit_history.columns:
    print(f"\n===== {c} =====")
    credit_history.select(c).distinct().show(20, False)

In [0]:
from pyspark.sql.functions import *

credit_history = spark.table(
    "credit_risk_analysis_db.bronze_sch.bronze_credit_history"
)

display(credit_history)

In [0]:
silver_credit_history = (

    credit_history

    .dropDuplicates()

    .filter(col("applicant_id").isNotNull())

    .withColumn(
        "credit_worthiness",
        upper(trim(col("credit_worthiness")))
    )

    .withColumn(
        "open_credit",
        upper(trim(col("open_credit")))
    )

    .withColumn(
        "credit_type",
        upper(trim(col("credit_type")))
    )

    .withColumn(
        "credit_score",
        col("credit_score").cast("double")
    )

    .withColumn(
        "co_applicant_credit_type",
        upper(trim(col("co_applicant_credit_type")))
    )

    .withColumn(
        "negative_amortization",
        upper(trim(col("negative_amortization")))
    )

    .withColumn(
        "interest_only",
        upper(trim(col("interest_only")))
    )

    .withColumn(
        "lump_sum_payment",
        upper(trim(col("lump_sum_payment")))
    )

    .withColumn(
        "debt_to_income_ratio",
        regexp_replace(col("debt_to_income_ratio"), "%", "").cast("double")
    )

    .withColumn(
        "credit_history_sk",
        md5(col("applicant_id"))
    )

    .withColumn(
        "effective_from",
        current_timestamp()
    )

    .withColumn(
        "effective_to",
        lit(None).cast("timestamp")
    )

    .withColumn(
        "is_current",
        lit(True)
    )

)

In [0]:
silver_credit_history.printSchema()

display(silver_credit_history)

In [0]:
(
    silver_credit_history.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(
        "credit_risk_analysis_db.silver_sch.silver_credit_history"
    )
)

In [0]:
display(
    spark.table(
        "credit_risk_analysis_db.silver_sch.silver_credit_history"
    )
)

In [0]:
loan_details = spark.table(
    "credit_risk_analysis_db.bronze_sch.bronze_loan_details"
)

loan_details.printSchema()

In [0]:
####4

In [0]:
from pyspark.sql.functions import *

loan_details = spark.table(
    "credit_risk_analysis_db.bronze_sch.bronze_loan_details"
)

display(loan_details)

In [0]:
cols = [
    "loan_amount",
    "loan_term",
    "property_value",
    "loan_to_value"
]

for c in cols:
    print(f"\n===== {c} =====")
    loan_details.select(c).distinct().show(20, False)

In [0]:
from pyspark.sql.functions import *

silver_loan_details = (

    loan_details

    .dropDuplicates()

    .filter(col("loan_id").isNotNull())

    .withColumn(
        "loan_type",
        upper(trim(col("loan_type")))
    )

    .withColumn(
        "loan_purpose",
        upper(trim(col("loan_purpose")))
    )

    .withColumn(
        "loan_amount",
        col("loan_amount").cast("double")
    )

    .withColumn(
        "loan_term",
        expr("try_cast(regexp_replace(trim(loan_term), '\\.0+$', '') AS int)")
    )

    .withColumn(
        "property_value",
        col("property_value").cast("double")
    )

    .withColumn(
        "loan_to_value",
        col("loan_to_value").cast("double")
    )

    .withColumn(
        "secured_by",
        upper(trim(col("secured_by")))
    )

    .withColumn(
        "security_type",
        upper(trim(col("security_type")))
    )

    # Surrogate Key
    .withColumn(
        "loan_sk",
        md5(col("loan_id"))
    )

    # SCD Type 2 Columns
    .withColumn(
        "effective_from",
        current_timestamp()
    )

    .withColumn(
        "effective_to",
        lit(None).cast("timestamp")
    )

    .withColumn(
        "is_current",
        lit(True)
    )

)

In [0]:
silver_loan_details.printSchema()

display(silver_loan_details)

In [0]:
(
    silver_loan_details.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(
        "credit_risk_analysis_db.silver_sch.silver_loan_details"
    )
)

In [0]:
display(
    spark.table(
        "credit_risk_analysis_db.silver_sch.silver_loan_details"
    )
)

In [0]:
economic = spark.table(
    "credit_risk_analysis_db.bronze_sch.bronze_economic_indicators"
)

economic.printSchema()

In [0]:
############5

In [0]:
from pyspark.sql.functions import *

economic = spark.table(
    "credit_risk_analysis_db.bronze_sch.bronze_economic_indicators"
)

display(economic)

In [0]:
cols = [
    "year",
    "avg_property_value",
    "avg_interest_rate",
    "Interest_rate_spread"
]

for c in cols:
    print(f"\n===== {c} =====")
    economic.select(c).distinct().show(20, False)

In [0]:
from pyspark.sql.functions import *

silver_economic = (

    economic

    .dropDuplicates()

    .filter(col("year").isNotNull())

    .withColumn(
        "year",
        expr("try_cast(year as int)")
    )

    .withColumn(
        "region",
        upper(trim(col("region")))
    )

    .withColumn(
        "avg_property_value",
        expr("try_cast(avg_property_value as double)")
    )

    .withColumn(
        "avg_interest_rate",
        expr("try_cast(avg_interest_rate as double)")
    )

    .withColumn(
        "Interest_rate_spread",
        expr("try_cast(Interest_rate_spread as double)")
    )

    .withColumn(
        "loan_status",
        upper(trim(col("loan_status")))
    )

    .withColumn(
        "economic_sk",
        md5(concat_ws("_", col("year"), col("region")))
    )

    .withColumn(
        "effective_from",
        current_timestamp()
    )

    .withColumn(
        "effective_to",
        lit(None).cast("timestamp")
    )

    .withColumn(
        "is_current",
        lit(True)
    )

)

In [0]:
silver_economic.printSchema()

display(silver_economic)

In [0]:
(
    silver_economic.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(
        "credit_risk_analysis_db.silver_sch.silver_economic_indicators"
    )
)

In [0]:
##############################################################################

In [0]:
from delta.tables import DeltaTable

target_table = "credit_risk_analysis_db.silver_sch.silver_applicant_profiles"

delta_table = DeltaTable.forName(spark, target_table)

source_df = spark.table(target_table)

In [0]:
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.applicant_id = source.applicant_id AND target.is_current = true"
    )
    .whenMatchedUpdate(
        condition="""
        target.income <> source.income OR
        target.region <> source.region
        """,
        set={
            "effective_to": "current_timestamp()",
            "is_current": "false"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
last_load = spark.sql("""
SELECT MAX(ingestion_timestamp)
FROM credit_risk_analysis_db.silver_sch.silver_applicant_profiles
""").collect()[0][0]

incremental_df = spark.table(
    "credit_risk_analysis_db.bronze_sch.bronze_applicant_profiles"
).filter(col("ingestion_timestamp") > lit(last_load))

In [0]:
%sql
OPTIMIZE credit_risk_analysis_db.silver_sch.silver_applicant_profiles;

In [0]:
%sql
OPTIMIZE credit_risk_analysis_db.silver_sch.silver_credit_application;

In [0]:
%sql
OPTIMIZE credit_risk_analysis_db.silver_sch.silver_credit_history;

In [0]:
%sql
OPTIMIZE credit_risk_analysis_db.silver_sch.silver_loan_details;

In [0]:
%sql
OPTIMIZE credit_risk_analysis_db.silver_sch.silver_economic_indicators;

In [0]:
%sql
OPTIMIZE credit_risk_analysis_db.silver_sch.silver_applicant_profiles
ZORDER BY (applicant_id);

OPTIMIZE credit_risk_analysis_db.silver_sch.silver_credit_application
ZORDER BY (applicant_id);

OPTIMIZE credit_risk_analysis_db.silver_sch.silver_credit_history
ZORDER BY (applicant_id);

OPTIMIZE credit_risk_analysis_db.silver_sch.silver_loan_details
ZORDER BY (loan_id);

OPTIMIZE credit_risk_analysis_db.silver_sch.silver_economic_indicators
ZORDER BY (year, region);

In [0]:
%sql
VACUUM credit_risk_analysis_db.silver_sch.silver_applicant_profiles RETAIN 168 HOURS;
VACUUM credit_risk_analysis_db.silver_sch.silver_credit_application RETAIN 168 HOURS;
VACUUM credit_risk_analysis_db.silver_sch.silver_credit_history RETAIN 168 HOURS;
VACUUM credit_risk_analysis_db.silver_sch.silver_loan_details RETAIN 168 HOURS;
VACUUM credit_risk_analysis_db.silver_sch.silver_economic_indicators RETAIN 168 HOURS;

In [0]:
%sql
DESCRIBE HISTORY credit_risk_analysis_db.silver_sch.silver_applicant_profiles;

In [0]:
%sql
SELECT *
FROM credit_risk_analysis_db.silver_sch.silver_applicant_profiles
VERSION AS OF 0;

In [0]:
%sql
SELECT *
FROM credit_risk_analysis_db.silver_sch.silver_applicant_profiles
TIMESTAMP AS OF '2026-07-09T19:07:26';